# 3단계: count 직접 예측 → 참여율(rate) 예측 전환

## 이 단계의 핵심 변화
인원 수를 직접 맞추는 대신 참여율을 먼저 예측하고 마지막에 인원으로 환산하는 구조로 바뀐 단계입니다.

## 왜 점수가 좋아졌나
가능 인원 대비 몇 %가 먹는가가 더 안정적인 패턴이어서 큰 개선 축이 되었습니다.

## 안내
이 파일은 다운로드 오류를 피하기 위해 새 이름으로 다시 만든 버전입니다.
코드 셀 안에는 기존 주석이 남아 있거나, 원본 설명 흐름을 유지합니다.


# 참여율 분모 후보 비교 노트북

목표:
- 식수 인원을 직접 예측하는 방식과
- **참여율(rate) 예측 후 식수 환산**하는 방식을 비교
- 특히 **분모(available 인원)를 어떻게 정의하느냐**에 따라 성능이 어떻게 달라지는지 확인

비교 방식:
- baseline: direct_count
- rate 모델: 각 분모 후보별 `rate_then_count`

중식 분모 후보:
1. `정원 - 휴가 - 출장`
2. `정원 - 휴가 - 출장 - 재택`
3. `정원 - 휴가 - 출장 - 0.5*재택`

석식 분모 후보:
1. `overtime`
2. `working_estimate + 0.5*overtime`
3. `0.2*working_estimate + overtime`
4. `working_estimate`

모델:
- 중식: CatBoost
- 석식: XGBoost

In [ ]:
# 필요시 한 번만 설치
# !pip install xgboost catboost scikit-learn

In [ ]:
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

warnings.filterwarnings("ignore")

mpl.rcParams.update(mpl.rcParamsDefault)
sns.reset_defaults()

available_fonts = sorted(set(f.name for f in fm.fontManager.ttflist))
font_candidates = [
    "NanumGothic", "NanumBarunGothic", "NanumSquare", "NanumSquareRound",
    "Noto Sans CJK KR", "Noto Sans KR", "Malgun Gothic", "AppleGothic"
]
selected_font = None
for font in font_candidates:
    if font in available_fonts:
        selected_font = font
        break

if selected_font is not None:
    mpl.rcParams["font.family"] = selected_font
    mpl.rcParams["font.sans-serif"] = [selected_font]
    mpl.rcParams["axes.unicode_minus"] = False
    sns.set_theme(style="whitegrid", rc={
        "font.family": selected_font,
        "font.sans-serif": [selected_font],
        "axes.unicode_minus": False,
    })
    print("적용 폰트:", selected_font)

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [ ]:
# ---------------------------
# 1. 파일 경로 자동 탐색
# ---------------------------
train_candidates = [
    r"C:\Users\yuzhd\github\team\data\made\train_median.csv",
    r"/mnt/c/Users/yuzhd/github/team/data/made/train_median.csv",
    r"/mnt/data/train_median.csv",
    r"./train_median.csv",
]

def find_existing_path(candidates):
    for p in candidates:
        if os.path.exists(p):
            return p
    return None

train_path = find_existing_path(train_candidates)
print("train path:", train_path)

if train_path is None:
    raise FileNotFoundError("train_median.csv 파일을 찾지 못했습니다.")

In [ ]:
# ---------------------------
# 2. 데이터 로드
# ---------------------------
df = pd.read_csv(train_path, encoding="utf-8-sig")
df["일자"] = pd.to_datetime(df["일자"])
df = df.sort_values("일자").reset_index(drop=True)

print("shape:", df.shape)
display(df.head())

In [ ]:
# ---------------------------
# 3. 기본 피처 엔지니어링
# ---------------------------
def add_base_features(df):
    df = df.copy()

    df["year"] = df["일자"].dt.year
    df["month"] = df["일자"].dt.month
    df["day"] = df["일자"].dt.day
    df["weekday_num"] = df["일자"].dt.weekday
    df["weekofyear"] = df["일자"].dt.isocalendar().week.astype(int)

    month_end = df["일자"] + pd.offsets.MonthEnd(0)
    df["days_to_month_end"] = (month_end - df["일자"]).dt.days

    df["is_month_start_part"] = (df["day"] <= 5).astype(int)
    df["is_month_end_part"] = (df["days_to_month_end"] <= 5).astype(int)
    df["is_wed"] = (df["weekday_num"] == 2).astype(int)
    df["is_fri"] = (df["weekday_num"] == 4).astype(int)
    df["is_covid"] = (df["일자"] >= pd.Timestamp("2020-01-01")).astype(int)

    df["working_estimate"] = (
        df["본사정원수"]
        - df["본사휴가자수"]
        - df["본사출장자수"]
        - df["현본사소속재택근무자수"]
    )

    df["absence_ratio"] = (
        df["본사휴가자수"] + df["본사출장자수"] + df["현본사소속재택근무자수"]
    ) / (df["본사정원수"] + 1)

    df["휴가자비율"] = df["본사휴가자수"] / (df["본사정원수"] + 1)
    df["출장자비율"] = df["본사출장자수"] / (df["본사정원수"] + 1)
    df["재택자비율"] = df["현본사소속재택근무자수"] / (df["본사정원수"] + 1)

    df["log_overtime"] = np.log1p(df["본사시간외근무명령서승인건수"])
    df["overtime_per_working"] = df["본사시간외근무명령서승인건수"] / (df["working_estimate"] + 1)
    df["has_overtime"] = (df["본사시간외근무명령서승인건수"] > 0).astype(int)

    if "석식메뉴" in df.columns:
        df["is_special_dinner_day"] = (df["석식메뉴"].astype(str) == "자기계발의날").astype(int)
    else:
        df["is_special_dinner_day"] = 0

    return df

df = add_base_features(df)

In [ ]:
# ---------------------------
# 4. 분모 후보 생성
# ---------------------------
eps = 1.0

# 중식 분모 후보
df["lunch_avail_v1"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"]).clip(lower=eps)
df["lunch_avail_v2"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - df["현본사소속재택근무자수"]).clip(lower=eps)
df["lunch_avail_v3"] = (df["본사정원수"] - df["본사휴가자수"] - df["본사출장자수"] - 0.5 * df["현본사소속재택근무자수"]).clip(lower=eps)

# 석식 분모 후보
df["dinner_avail_v1"] = df["본사시간외근무명령서승인건수"].clip(lower=eps)
df["dinner_avail_v2"] = (df["working_estimate"] + 0.5 * df["본사시간외근무명령서승인건수"]).clip(lower=eps)
df["dinner_avail_v3"] = (0.2 * df["working_estimate"] + df["본사시간외근무명령서승인건수"]).clip(lower=eps)
df["dinner_avail_v4"] = df["working_estimate"].clip(lower=eps)

display(df[[
    "일자", "중식계", "석식계",
    "lunch_avail_v1", "lunch_avail_v2", "lunch_avail_v3",
    "dinner_avail_v1", "dinner_avail_v2", "dinner_avail_v3", "dinner_avail_v4"
]].head())

In [ ]:
# ---------------------------
# 5. 피처 구성
# ---------------------------
feature_cols = [
    "본사정원수","본사휴가자수","본사출장자수","본사시간외근무명령서승인건수",
    "현본사소속재택근무자수","year","month","day","weekday_num","weekofyear",
    "days_to_month_end","is_month_start_part","is_month_end_part","is_wed","is_fri",
    "is_covid","working_estimate","absence_ratio","휴가자비율","출장자비율",
    "재택자비율","log_overtime","overtime_per_working","has_overtime","is_special_dinner_day",
]

split_date = pd.Timestamp("2020-11-01")
train_df = df[df["일자"] < split_date].copy()
valid_df = df[df["일자"] >= split_date].copy()

X_train = train_df[feature_cols].copy()
X_valid = valid_df[feature_cols].copy()

print("train:", train_df.shape)
print("valid:", valid_df.shape)

In [ ]:
# ---------------------------
# 6. 평가 함수
# ---------------------------
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

def nmae_fixed(y_true, y_pred):
    return mean_absolute_error(y_true, y_pred) / (np.mean(y_true) + 1e-8) * 100

def evaluate_regression(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "NMAE_fixed(%)": nmae_fixed(y_true, y_pred),
    }

In [ ]:
# ---------------------------
# 7. baseline: 직접 식수 예측
# ---------------------------
lunch_direct_model = CatBoostRegressor(
    iterations=500, learning_rate=0.05, depth=6,
    random_state=42, verbose=0
)
lunch_direct_model.fit(X_train, train_df["중식계"])
pred_lunch_direct = lunch_direct_model.predict(X_valid)

dinner_direct_model = XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.9, colsample_bytree=0.9,
    objective="reg:squarederror", random_state=42
)
dinner_direct_model.fit(X_train, train_df["석식계"])
pred_dinner_direct = dinner_direct_model.predict(X_valid)

baseline_rows = [
    {"target":"중식계", "candidate":"direct_count", **evaluate_regression(valid_df["중식계"].values, pred_lunch_direct)},
    {"target":"석식계", "candidate":"direct_count", **evaluate_regression(valid_df["석식계"].values, pred_dinner_direct)},
]
baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df)

In [ ]:
# ---------------------------
# 8. 중식 분모 후보 비교
# ---------------------------
lunch_candidates = ["lunch_avail_v1", "lunch_avail_v2", "lunch_avail_v3"]
lunch_rows = []

for cand in lunch_candidates:
    train_rate = (train_df["중식계"] / train_df[cand]).clip(lower=0, upper=1.5)
    valid_avail = valid_df[cand].values

    model = CatBoostRegressor(
        iterations=500, learning_rate=0.05, depth=6,
        random_state=42, verbose=0
    )
    model.fit(X_train, train_rate)

    pred_rate = model.predict(X_valid)
    pred_rate = np.clip(pred_rate, 0, 1.5)
    pred_count = pred_rate * valid_avail

    lunch_rows.append({
        "target":"중식계",
        "candidate":cand,
        **evaluate_regression(valid_df["중식계"].values, pred_count)
    })

lunch_rate_df = pd.DataFrame(lunch_rows).sort_values(["NMAE_fixed(%)", "RMSE"]).reset_index(drop=True)
display(lunch_rate_df)

In [ ]:
# ---------------------------
# 9. 석식 분모 후보 비교
# ---------------------------
dinner_candidates = ["dinner_avail_v1", "dinner_avail_v2", "dinner_avail_v3", "dinner_avail_v4"]
dinner_rows = []

for cand in dinner_candidates:
    train_rate = (train_df["석식계"] / train_df[cand]).clip(lower=0, upper=1.5)
    valid_avail = valid_df[cand].values

    model = XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=6,
        subsample=0.9, colsample_bytree=0.9,
        objective="reg:squarederror", random_state=42
    )
    model.fit(X_train, train_rate)

    pred_rate = model.predict(X_valid)
    pred_rate = np.clip(pred_rate, 0, 1.5)
    pred_count = pred_rate * valid_avail

    dinner_rows.append({
        "target":"석식계",
        "candidate":cand,
        **evaluate_regression(valid_df["석식계"].values, pred_count)
    })

dinner_rate_df = pd.DataFrame(dinner_rows).sort_values(["NMAE_fixed(%)", "RMSE"]).reset_index(drop=True)
display(dinner_rate_df)

In [ ]:
# ---------------------------
# 10. 전체 비교표
# ---------------------------
all_compare_df = pd.concat([baseline_df, lunch_rate_df, dinner_rate_df], axis=0).reset_index(drop=True)
display(all_compare_df)

best_lunch = lunch_rate_df.iloc[0:1].copy()
best_dinner = dinner_rate_df.iloc[0:1].copy()

print("[중식 best candidate]")
display(best_lunch)

print("[석식 best candidate]")
display(best_dinner)

In [ ]:
# ---------------------------
# 11. 시각화
# ---------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

lunch_plot = pd.concat([
    baseline_df[baseline_df["target"] == "중식계"],
    lunch_rate_df
], axis=0)

dinner_plot = pd.concat([
    baseline_df[baseline_df["target"] == "석식계"],
    dinner_rate_df
], axis=0)

sns.barplot(data=lunch_plot, x="candidate", y="NMAE_fixed(%)", ax=axes[0])
axes[0].set_title("중식 분모 후보 비교")
axes[0].tick_params(axis="x", rotation=25)

sns.barplot(data=dinner_plot, x="candidate", y="NMAE_fixed(%)", ax=axes[1])
axes[1].set_title("석식 분모 후보 비교")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------
# 12. 결과 저장
# ---------------------------
baseline_df.to_csv("participation_rate_baseline_compare.csv", index=False, encoding="utf-8-sig")
lunch_rate_df.to_csv("participation_rate_lunch_candidates.csv", index=False, encoding="utf-8-sig")
dinner_rate_df.to_csv("participation_rate_dinner_candidates.csv", index=False, encoding="utf-8-sig")
all_compare_df.to_csv("participation_rate_candidates_summary.csv", index=False, encoding="utf-8-sig")

print("저장 완료:")
print("- participation_rate_baseline_compare.csv")
print("- participation_rate_lunch_candidates.csv")
print("- participation_rate_dinner_candidates.csv")
print("- participation_rate_candidates_summary.csv")

In [ ]:
# ---------------------------
# 13. 해석 가이드
# ---------------------------
print(
    "해석 포인트:\n"
    "1. direct_count보다 더 낮은 NMAE_fixed(%)를 보이는 분모 후보가 있으면 참여율 접근이 유효할 가능성이 있습니다.\n"
    "2. 중식과 석식은 서로 다른 분모 정의가 필요할 가능성이 높습니다.\n"
    "3. 특히 석식은 overtime 중심 분모가 유효할 수 있으니 dinner_avail_v1~v3 결과를 주의 깊게 보세요."
)